# Análise Completa de Features - Dataset DoS (MQTT Under Attack)

Este notebook realiza uma análise detalhada de todas as 67 features do dataset DoS, incluindo:

1. **Identificação de Tipos**: Tipos de dados de cada feature
2. **Análise de Valores Ausentes**: Features com dados faltantes
3. **Análise de Variância**: Features com pouca ou nenhuma variação
4. **Distribuição de Classes**: Análise do target `type`
5. **Classificação de Relevância**: Features úteis vs inúteis para detecção de DoS
6. **Recomendações**: Features mais bem avaliadas por múltiplos métodos de seleção

**Fonte do Dataset**: [MQTT_UAD - MQTT Under Attack Dataset](https://figshare.com/articles/dataset/MQTT_UAD_MQTT_Under_Attack_Dataset_A_public_dataset_for_the_detection_of_attacks_in_IoT_networks_using_MQTT_protocol/24420958)

## 1. Configuração e Carregamento dos Dados

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

print("Bibliotecas carregadas com sucesso!")

In [ ]:
# Carregar o dataset DoS
dos_path = Path("../data/raw/MQTT Under Attack Dataset/DoS.csv")
df = pd.read_csv(dos_path)

print(f"Shape do dataset: {df.shape}")
print(f"Colunas: {df.shape[1]}")
print(f"Registros: {df.shape[0]}")
print(f"\nColunas do Dataset:")
for i, col in enumerate(df.columns, 1):
    print(f"  {i:2d}. {col}")

## 2. Visão Geral do Dataset

In [ ]:
# Primeiras linhas do dataset
df.head()

In [ ]:
# Informações gerais sobre tipos e valores ausentes
df.info()

In [ ]:
# Distribuição da variável target (type)
print("Distribuição do Label (type):")
print(df['type'].value_counts())
print(f"\nProporções:")
print(df['type'].value_counts(normalize=True).round(4) * 100)

In [ ]:
# Visualização da distribuição de classes
fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#2ca02c' if x == 'normal' else '#d62728' for x in df['type'].value_counts().index]
df['type'].value_counts().plot(kind='bar', ax=ax, color=colors, edgecolor='black')
ax.set_xlabel('Tipo de Tráfego')
ax.set_ylabel('Quantidade de Registros')
ax.set_title('Distribuição das Classes no Dataset DoS')
ax.tick_params(axis='x', rotation=0)

# Adicionar valores nas barras
for i, v in enumerate(df['type'].value_counts().values):
    ax.text(i, v + 500, f'{v:,}', ha='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

## 3. Análise de Tipos de Dados

In [ ]:
# Criar DataFrame com informações detalhadas de cada coluna
def analyze_columns(df):
    analysis = []
    for col in df.columns:
        col_data = {
            'Coluna': col,
            'Tipo_Pandas': str(df[col].dtype),
            'Valores_Unicos': df[col].nunique(),
            'Valores_Nulos': df[col].isna().sum(),
            'Pct_Nulos': (df[col].isna().sum() / len(df) * 100).round(2),
            'Exemplo_Valores': str(df[col].dropna().head(3).tolist())[:50]
        }
        
        # Determinar tipo semântico
        if df[col].dtype in ['int64', 'float64']:
            col_data['Tipo_Semantico'] = 'Numérico'
        elif df[col].dtype == 'object':
            if df[col].nunique() < 20:
                col_data['Tipo_Semantico'] = 'Categórico'
            else:
                col_data['Tipo_Semantico'] = 'Texto'
        else:
            col_data['Tipo_Semantico'] = 'Outro'
        
        analysis.append(col_data)
    
    return pd.DataFrame(analysis)

col_analysis = analyze_columns(df)
col_analysis

In [ ]:
# Resumo por tipo de dado
print("Resumo por Tipo Semântico:")
print(col_analysis['Tipo_Semantico'].value_counts())

print("\nResumo por Tipo Pandas:")
print(col_analysis['Tipo_Pandas'].value_counts())

## 4. Análise de Valores Ausentes

In [ ]:
# Features com valores ausentes
missing_cols = col_analysis[col_analysis['Valores_Nulos'] > 0].sort_values('Pct_Nulos', ascending=False)

print(f"Features com valores ausentes: {len(missing_cols)}/{len(df.columns)}")
print(f"\nDetalhes:")
missing_cols[['Coluna', 'Valores_Nulos', 'Pct_Nulos']]

In [ ]:
# Visualização de valores ausentes (top 30)
if len(missing_cols) > 0:
    fig, ax = plt.subplots(figsize=(12, 8))
    top_missing = missing_cols.head(30)
    
    colors = plt.cm.Reds(top_missing['Pct_Nulos'].values / 100)
    bars = ax.barh(range(len(top_missing)), top_missing['Pct_Nulos'].values, color=colors)
    ax.set_yticks(range(len(top_missing)))
    ax.set_yticklabels(top_missing['Coluna'].values)
    ax.set_xlabel('% de Valores Ausentes')
    ax.set_title('Top 30 Features com Valores Ausentes')
    ax.invert_yaxis()
    
    for bar, pct in zip(bars, top_missing['Pct_Nulos'].values):
        ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
                f'{pct:.1f}%', va='center', fontsize=9)
    
    plt.tight_layout()
    plt.show()

## 5. Análise de Variância e Features Constantes

In [ ]:
# Identificar features numéricas
num_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
print(f"Features numéricas: {len(num_cols)}")

# Calcular variância
variance_df = pd.DataFrame({
    'Feature': num_cols,
    'Variancia': df[num_cols].var(),
    'Valores_Unicos': df[num_cols].nunique(),
    'Min': df[num_cols].min(),
    'Max': df[num_cols].max()
}).sort_values('Variancia')

# Features com variância zero ou muito baixa
low_variance = variance_df[variance_df['Valores_Unicos'] <= 1]
print(f"\nFeatures constantes (valor único ou apenas nulos): {len(low_variance)}")
low_variance

In [ ]:
# Features com poucos valores únicos (quasi-constantes)
quasi_constant = variance_df[(variance_df['Valores_Unicos'] > 1) & (variance_df['Valores_Unicos'] <= 3)]
print(f"Features quasi-constantes (2-3 valores únicos): {len(quasi_constant)}")
quasi_constant

## 6. Classificação das Features por Grupo

In [ ]:
# Classificar features por grupo funcional
feature_classification = {
    'Frame/Captura': [
        'frame.time_delta', 'frame.time_delta_displayed', 'frame.time_epoch',
        'frame.time_invalid', 'frame.time_relative', 'frame.cap_len',
        'frame.coloring_rule.name', 'frame.coloring_rule.string', 'frame.comment',
        'frame.comment.expert', 'frame.encap_type', 'frame.file_off',
        'frame.ignored', 'frame.incomplete', 'frame.interface_id',
        'frame.interface_name', 'frame.len', 'frame.link_nr',
        'frame.marked', 'frame.md5_hash', 'frame.number', 'frame.offset_shift'
    ],
    'Rede (IP/TCP)': [
        'ip.src', 'ip.dst', 'tcp.srcport', 'tcp.dstport', 'eth.src', 'eth.dst'
    ],
    'MQTT - Conexão': [
        'mqtt.clientid', 'mqtt.clientid_len', 'mqtt.conack.flags',
        'mqtt.conack.flags.reserved', 'mqtt.conack.flags.sp', 'mqtt.conack.val',
        'mqtt.conflag.cleansess', 'mqtt.conflag.passwd', 'mqtt.conflag.qos',
        'mqtt.conflag.reserved', 'mqtt.conflag.retain', 'mqtt.conflag.uname',
        'mqtt.conflag.willflag', 'mqtt.conflags', 'mqtt.kalive',
        'mqtt.proto_len', 'mqtt.protoname', 'mqtt.ver'
    ],
    'MQTT - Mensagens': [
        'mqtt.dupflag', 'mqtt.hdrflags', 'mqtt.len', 'mqtt.msg',
        'mqtt.msgid', 'mqtt.msgtype', 'mqtt.qos', 'mqtt.retain',
        'mqtt.topic', 'mqtt.topic_len'
    ],
    'MQTT - Autenticação': [
        'mqtt.username', 'mqtt.username_len', 'mqtt.passwd', 'mqtt.passwd_len'
    ],
    'MQTT - Assinatura': [
        'mqtt.sub.qos', 'mqtt.suback.qos'
    ],
    'MQTT - Last Will': [
        'mqtt.willmsg', 'mqtt.willmsg_len', 'mqtt.willtopic', 'mqtt.willtopic_len'
    ],
    'Target': ['type']
}

# Criar tabela de classificação
classification_list = []
for group, features in feature_classification.items():
    for feat in features:
        if feat in df.columns:
            classification_list.append({'Feature': feat, 'Grupo': group})

classification_df = pd.DataFrame(classification_list)
print("Classificação das Features por Grupo:")
print(classification_df['Grupo'].value_counts())

In [ ]:
# Visualização da distribuição por grupo
fig, ax = plt.subplots(figsize=(10, 6))
group_counts = classification_df['Grupo'].value_counts()
colors = plt.cm.Set3(np.linspace(0, 1, len(group_counts)))

bars = ax.barh(range(len(group_counts)), group_counts.values, color=colors)
ax.set_yticks(range(len(group_counts)))
ax.set_yticklabels(group_counts.index)
ax.set_xlabel('Número de Features')
ax.set_title('Distribuição das Features por Grupo Funcional')
ax.invert_yaxis()

for bar, count in zip(bars, group_counts.values):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
            f'{count}', va='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

## 7. Análise de Relevância para Detecção de DoS

In [ ]:
# Features recomendadas pelos métodos de seleção (resultado do notebook matriz_correlacao)
# Baseado no consenso de 6 métodos: mRMR, Fisher, Pearson, ExtraTrees, LinearSVC, Lasso

features_6_metodos = [
    'frame.cap_len',
    'frame.len',
    'mqtt.clientid_len',
    'mqtt.qos',
    'mqtt.conack.flags.reserved',
    'mqtt.len',
    'mqtt.topic_len',
    'mqtt.kalive',
    'mqtt.msgtype'
]

features_5_metodos = [
    'mqtt.retain',
    'mqtt.conflag.cleansess',
    'mqtt.proto_len'
]

features_4_metodos = [
    'frame.time_delta',
    'mqtt.conack.val',
    'mqtt.ver'
]

print("FEATURES MAIS BEM AVALIADAS PARA DETECÇÃO DE DoS")
print("="*60)
print(f"\n[ALTA RELEVÂNCIA] Selecionadas por 6/6 métodos ({len(features_6_metodos)} features):")
for f in features_6_metodos:
    print(f"   * {f}")

print(f"\n[MÉDIA-ALTA RELEVÂNCIA] Selecionadas por 5/6 métodos ({len(features_5_metodos)} features):")
for f in features_5_metodos:
    print(f"   * {f}")

print(f"\n[MÉDIA RELEVÂNCIA] Selecionadas por 4/6 métodos ({len(features_4_metodos)} features):")
for f in features_4_metodos:
    print(f"   * {f}")

In [ ]:
# Estatísticas das features de alta relevância
high_relevance_features = features_6_metodos + features_5_metodos + features_4_metodos
print("Estatísticas das Features de Alta Relevância:")
df[high_relevance_features].describe()

## 8. Features Provavelmente Inúteis ou Irrelevantes

In [ ]:
# Identificar features provavelmente inúteis
features_inuteis = {
    'Identificadores de Captura (Metadados Wireshark)': {
        'features': [
            'frame.time_epoch',           # Timestamp absoluto
            'frame.time_relative',        # Tempo relativo à captura
            'frame.time_delta_displayed', # Igual a frame.time_delta
            'frame.file_off',             # Offset no arquivo
            'frame.number',               # Número sequencial do pacote
            'frame.offset_shift',         # Deslocamento de offset
            'frame.md5_hash',             # Hash do frame
            'frame.interface_id',         # ID da interface de captura
            'frame.interface_name',       # Nome da interface
            'frame.encap_type',           # Tipo de encapsulamento (constante)
            'frame.link_nr',              # Número do link
        ],
        'razao': 'Metadados de captura do Wireshark, não relacionados ao tráfego MQTT'
    },
    'Features de Anotação/Visualização': {
        'features': [
            'frame.coloring_rule.name',   # Regra de coloração
            'frame.coloring_rule.string', # String da regra
            'frame.comment',              # Comentários do usuário
            'frame.comment.expert',       # Comentários especializados
            'frame.marked',               # Frame marcado
            'frame.ignored',              # Frame ignorado
        ],
        'razao': 'Anotações manuais da ferramenta de análise, não contêm informação do tráfego'
    },
    'Features Constantes ou Quasi-Constantes': {
        'features': [
            'frame.time_invalid',         # Sempre nulo ou 0
            'frame.incomplete',           # Sempre 0
            'mqtt.conflag.reserved',      # Sempre 0 (reservado pelo protocolo)
            'mqtt.conack.flags',          # Redundante com flags específicos
        ],
        'razao': 'Variância zero ou muito baixa - não discriminam entre classes'
    },
    'Identificadores de Endereço': {
        'features': [
            'ip.src',                     # IP de origem
            'ip.dst',                     # IP de destino
            'eth.src',                    # MAC de origem
            'eth.dst',                    # MAC de destino
        ],
        'razao': 'Identificadores específicos do ambiente de teste - causam overfitting'
    },
    'Portas TCP (Parcialmente Relevantes)': {
        'features': [
            'tcp.srcport',                # Porta de origem (geralmente efêmera)
            'tcp.dstport',                # Porta de destino (geralmente 1883 para MQTT)
        ],
        'razao': 'Portas podem ser fixas no dataset - verificar se há variação suficiente'
    },
    'Conteúdo de Texto (Identificadores)': {
        'features': [
            'mqtt.clientid',              # ID do cliente MQTT
            'mqtt.username',              # Nome de usuário
            'mqtt.passwd',                # Senha
            'mqtt.msg',                   # Conteúdo da mensagem
            'mqtt.topic',                 # Nome do tópico
            'mqtt.protoname',             # Nome do protocolo (sempre "MQTT")
            'mqtt.willmsg',               # Mensagem Last Will
            'mqtt.willtopic',             # Tópico Last Will
        ],
        'razao': 'Dados textuais que requerem encoding especial; identificadores podem causar overfitting'
    },
    'Features Redundantes': {
        'features': [
            'frame.cap_len',              # Altamente correlacionado com frame.len
            'mqtt.conflags',              # Composição binária dos conflag.* individuais
            'mqtt.hdrflags',              # Composição de flags do header
        ],
        'razao': 'Redundantes com outras features mais específicas'
    }
}

print("FEATURES PROVAVELMENTE INÚTEIS OU DE BAIXA RELEVÂNCIA")
print("="*70)

total_inuteis = 0
for categoria, info in features_inuteis.items():
    features_existentes = [f for f in info['features'] if f in df.columns]
    total_inuteis += len(features_existentes)
    print(f"\n[{categoria}] ({len(features_existentes)} features)")
    print(f"  Razão: {info['razao']}")
    print(f"  Features:")
    for f in features_existentes:
        print(f"    - {f}")

print(f"\n{'='*70}")
print(f"Total de features potencialmente inúteis: {total_inuteis}/{len(df.columns)}")

In [ ]:
# Verificar features que não foram selecionadas por NENHUM método
# Carregar ranking de consenso
try:
    consensus_df = pd.read_csv('../reports/feature_consensus_ranking.csv')
    features_selecionadas = set(consensus_df['Feature'].tolist())
    
    # Features numéricas que não foram selecionadas por nenhum método
    num_features = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
    nao_selecionadas = [f for f in num_features if f not in features_selecionadas]
    
    print(f"Features numéricas não selecionadas por nenhum dos 6 métodos: {len(nao_selecionadas)}")
    for f in nao_selecionadas:
        print(f"  - {f}")
except FileNotFoundError:
    print("Arquivo de ranking não encontrado. Execute o notebook matriz_correlacao.ipynb primeiro.")

## 9. Resumo Final: Classificação de Features

In [ ]:
# Criar resumo consolidado
def classificar_relevancia(feature):
    if feature in features_6_metodos:
        return 'ALTA (6/6 métodos)'
    elif feature in features_5_metodos:
        return 'MÉDIA-ALTA (5/6 métodos)'
    elif feature in features_4_metodos:
        return 'MÉDIA (4/6 métodos)'
    
    # Verificar se é informação de identificação
    for categoria, info in features_inuteis.items():
        if feature in info['features']:
            return f'BAIXA/INÚTIL - {categoria}'
    
    return 'NÃO AVALIADA ou BAIXA'

# Criar DataFrame final
resumo_features = []
for col in df.columns:
    resumo_features.append({
        'Feature': col,
        'Tipo': str(df[col].dtype),
        'Valores_Unicos': df[col].nunique(),
        'Pct_Nulos': round(df[col].isna().sum() / len(df) * 100, 2),
        'Relevancia': classificar_relevancia(col)
    })

resumo_df = pd.DataFrame(resumo_features)

# Ordenar por relevância
ordem_relevancia = {
    'ALTA (6/6 métodos)': 1,
    'MÉDIA-ALTA (5/6 métodos)': 2,
    'MÉDIA (4/6 métodos)': 3,
    'NÃO AVALIADA ou BAIXA': 4
}
resumo_df['Ordem'] = resumo_df['Relevancia'].apply(
    lambda x: ordem_relevancia.get(x, 5)
)
resumo_df = resumo_df.sort_values('Ordem')

print("RESUMO FINAL DE CLASSIFICAÇÃO DAS FEATURES")
print("="*80)
resumo_df[['Feature', 'Tipo', 'Valores_Unicos', 'Pct_Nulos', 'Relevancia']]

In [ ]:
# Salvar resumo em CSV
output_path = Path('../reports/feature_analysis_dos.csv')
resumo_df[['Feature', 'Tipo', 'Valores_Unicos', 'Pct_Nulos', 'Relevancia']].to_csv(output_path, index=False)
print(f"Resumo salvo em: {output_path}")

In [ ]:
# Contagem por categoria de relevância
print("\nDistribuição por Relevância:")
print(resumo_df['Relevancia'].value_counts())

## 10. Conclusões e Recomendações

### Features Recomendadas para Modelo de Detecção de DoS

Com base na análise de 6 métodos de seleção de features (mRMR, Fisher's Score, Pearson, ExtraTrees, LinearSVC, Lasso), as seguintes features são **altamente recomendadas**:

| Feature | Descrição | Métodos |
|---------|-----------|--------|
| `frame.cap_len` | Tamanho do frame capturado | 6/6 |
| `frame.len` | Tamanho do frame | 6/6 |
| `mqtt.clientid_len` | Comprimento do Client ID | 6/6 |
| `mqtt.qos` | Quality of Service | 6/6 |
| `mqtt.conack.flags.reserved` | Flags reservados CONNACK | 6/6 |
| `mqtt.len` | Tamanho da mensagem MQTT | 6/6 |
| `mqtt.topic_len` | Comprimento do tópico | 6/6 |
| `mqtt.kalive` | Keep Alive | 6/6 |
| `mqtt.msgtype` | Tipo de mensagem MQTT | 6/6 |

### Features a Descartar

- **Metadados de captura**: frame.time_epoch, frame.number, frame.file_off, etc.
- **Identificadores de rede**: ip.src, ip.dst, eth.src, eth.dst (causam overfitting)
- **Features constantes**: frame.time_invalid, mqtt.conflag.reserved
- **Conteúdo textual**: mqtt.msg, mqtt.topic, mqtt.clientid (requerem tratamento especial)